# Демо: разделение речи двух дикторов

# 1. Репозиторий и окружение

In [ ]:
REPO = "https://github.com/ndromanenko/avss.git"
BRANCH = "main"

!git clone -b {BRANCH} {REPO} avss
%cd avss

In [ ]:
%pip install -q uv
!uv python install 3.12
!uv sync --python 3.12 --locked

# 2. Параметры

`DATA_DIR`: папка формата CustomDir:

In [ ]:
%env MPLBACKEND=Agg

from pathlib import Path
from IPython.display import Audio, display

DATA_DIR = "data/test_with_targets"
CKPT = "ckpt/model_best.pth"
USE_VIDEO = True # False для аудиомодели
SAVE_PATH = "predictions"

# 3. Модель и данные

In [ ]:
!uv run python download_model.py

In [ ]:
DATASET_URL = ""  # ссылка на zip с папкой формата CustomDir

if DATASET_URL:
    !uv run --with gdown gdown "{DATASET_URL}" -O data.zip
    !unzip -q -o data.zip -d data

# 4. Инференс

In [ ]:
tensors = ["mix"]
if Path(DATA_DIR, "audio", "s1").exists():
    tensors.append("targets")
if USE_VIDEO:
    tensors.append("video")
print("Что будет на device")
print(tensors)
DEVICE_TENSORS = "[" + ",".join(tensors) + "]"

!uv run inference.py \
    datasets.test.dir={DATA_DIR} \
    inferencer.from_pretrained={CKPT} \
    inferencer.save_path={SAVE_PATH} \
    'inferencer.device_tensors={DEVICE_TENSORS}' \
    dataloader.batch_size=1 dataloader.num_workers=0

Пример разделения: смесь и 2 восстановленные дорожки.

In [ ]:
name = sorted(Path(DATA_DIR, "audio", "mix").iterdir())[0].name
print(name)
display(Audio(f"{DATA_DIR}/audio/mix/{name}"))
display(Audio(f"{SAVE_PATH}/test/s1/{name}"))
display(Audio(f"{SAVE_PATH}/test/s2/{name}"))

# 5. Метрики
Требуются эталонные дорожки `audio/s1` и `audio/s2`.

In [ ]:
!uv run calc_metrics.py target_dir={DATA_DIR}/audio pred_dir={SAVE_PATH}/test

# 6. Эффективность

In [ ]:
!uv run measure_efficiency.py checkpoint={CKPT} data_dir={DATA_DIR}